This notebook largely follows the approach of ["An NLP-based approach to assessing a company's maturity level in the digital era" (2024) by Romano, Sperlì, and Vignali](https://www.sciencedirect.com/science/article/pii/S0957417424011588)

Extra pre-requisites:
* Create a Hugging Face (HF) account
* Create a HF access token with at least read access
* Save the token value within Google Colab secrets (secret name: "HF_TOKEN") or within ??? if running this notebook locally.

In [3]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [1]:
import math
import matplotlib
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
import string
import sys
import transformers

In [2]:
import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import PunktSentenceTokenizer, RegexpTokenizer

NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Unzipping corpora/abc.zip.
       | Downloading package alpino to /root/nltk_data...
       |   Unzipping corpora/alpino.zip.
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger.zip.
       | Downloading package averaged_perceptron_tagger_eng to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_ru.zip.
       | Downloading package averaged_perceptron_tagger_rus to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_rus.zip.
       | Downloading package basque_grammars to /root/nltk_data...
       |   Unzipping grammars/basque_grammars.zip.
       | Downloading package bcp47 to /root/nltk_d


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> q


In [4]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

from innoprod.plotting_tools import rand_jitter
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.text_analysis.chunking_tools import chunk_text_sentencewise
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

Cloning into 'digi-inno-road-prod'...
remote: Enumerating objects: 485, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 485 (delta 75), reused 66 (delta 31), pack-reused 360 (from 1)
Receiving objects: 100% (485/485), 916.69 KiB | 7.64 MiB/s, done.
Resolving deltas: 100% (284/284), done.


In [5]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

/content/digi-inno-road-prod/innoprod/wrangling/wrangling_tools.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series.loc[mask] = new_value


In [30]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    # 'Key potential industry 4.0 solutions to address the identified problems/gaps',
    # 'Recommended Action Plan to utilise Industry 4.0 Technology',
    # 'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    # 'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    # 'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

In [31]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

,Client ID,Question,Response
0,5a43e5a6-49e2-3bf5-0b8a-66743ea7166c,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 5 STATUS: Based on your resp...
1,E2413853-2C45-52BD-D6BD-0822CBF064BA,Summary review of Edge Digital diagnostic repo...,The management at AET has invested significant...
2,5f927abf-d217-9b9e-e524-666aa04e0984,Summary review of Edge Digital diagnostic repo...,The management at [REDACTED] has invested sign...
3,CE3AC64A-B69B-5045-DDE7-65B572468FC9,Summary review of Edge Digital diagnostic repo...,[REDACTED]Limited benchmarks well against its ...
4,2F391E8E-8BE8-1A59-68FF-D02CFF026390,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 5 STATUS: Based on your resp...
...,...,...,...
1315,b6060463-e942-8a4a-77be-67caff694cdf,"Summary of the identified problems, including ...","The high level of investment required, extende..."
1316,8bf25d49-f732-82c8-1274-67dd87ab56e9,"Summary of the identified problems, including ...",Frazers currently have no digitisation in plac...
1317,63ebf90a-ed7d-e34f-4033-67b30f6d19f4,"Summary of the identified problems, including ...",Understanding and communicating the businesses...
1318,8dbafede-6e05-b0ed-af96-67dd8480f482,"Summary of the identified problems, including ...",The company should consider support in the onw...


In [32]:
sent_tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: sent_tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

,Sentence Number,Client ID,Question,Sentence
0,1,5a43e5a6-49e2-3bf5-0b8a-66743ea7166c,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 5 STATUS: Based on your resp...
1,2,5a43e5a6-49e2-3bf5-0b8a-66743ea7166c,Summary review of Edge Digital diagnostic repo...,You can build on this strong foundation to boo...
2,3,5a43e5a6-49e2-3bf5-0b8a-66743ea7166c,Summary review of Edge Digital diagnostic repo...,RECOMMENDATIONS: Work with your [REDACTED] Adv...
3,4,5a43e5a6-49e2-3bf5-0b8a-66743ea7166c,Summary review of Edge Digital diagnostic repo...,NEXT STEPS: Review your current strategy and r...
4,1,E2413853-2C45-52BD-D6BD-0822CBF064BA,Summary review of Edge Digital diagnostic repo...,The management at AET has invested significant...
...,...,...,...,...
4199,3,c52bafa5-941e-1e2e-ddf0-67cafecce415,"Summary of the identified problems, including ...",Beyond this a client data management system wh...
4200,4,c52bafa5-941e-1e2e-ddf0-67cafecce415,"Summary of the identified problems, including ...",It is expected that this increased functionali...
4201,5,c52bafa5-941e-1e2e-ddf0-67cafecce415,"Summary of the identified problems, including ...",The expectation is that this could drive sales...
4202,6,c52bafa5-941e-1e2e-ddf0-67cafecce415,"Summary of the identified problems, including ...",In addition to this investment in skills devel...


# Preprocessing

In [33]:
regexpr = "[\\S]+"
re_tokenizer = RegexpTokenizer(regexpr)

all_text = responses_df['Response'].str.cat(sep=" ")
all_words = re_tokenizer.tokenize(all_text)
all_words

needs_assessing = [False]*len(all_words)

for i, word in enumerate(all_words):
  if "." not in word:
    continue
  # If there's only one "." in the word and it's at the end, then it's probably
  # just the end of the sentence. Check if the next word starts with an upper-
  # case character (if there is a next word).
  if word.count(".") == 1 and word.index(".") == len(word)-1:
    if (i == len(all_words)-1) or all_words[i+1][0].isupper():
      continue
  needs_assessing[i] = True

words_to_assess = set([word for i, word in enumerate(all_words) if needs_assessing[i]])
words_to_assess


{'"anti-IT".',
 '(i.e.',
 '.',
 '.-Big',
 '.[REDACTED]',
 '1.',
 '2.',
 '3.',
 '4.',
 '4.0',
 '4.0,',
 '4.0.',
 '4.o',
 '5.',
 'A.Bush',
 'A.[REDACTED]',
 'AI.',
 'E.O.L',
 'Floor).',
 'I4.0',
 'Ind.',
 'Industry4.0',
 'LBBC.',
 'ROI.',
 'SAGE.',
 '[REDACTED].',
 'above.',
 'accessible.',
 'activities.',
 'activity.',
 'addressed.',
 'adopted.',
 'adoption.',
 'advances.The',
 'again.',
 'agriculture.',
 'applicable.',
 'appropriate.',
 'area.',
 'areas.',
 'aspirations.',
 'assessed.There',
 'attained.',
 'automation.',
 'available.',
 'barrier.',
 'barriers.',
 'base.',
 'basis.',
 'bespoke.',
 'better.',
 'board.',
 'bodies.',
 'breach.',
 'bring.',
 'business.',
 'businesses.',
 'capacity.',
 'carefully.',
 'causes.',
 'challenge.',
 'challenges.',
 'challenging.',
 'change.',
 'checking.',
 'clients.',
 'competencies.',
 'competitive.',
 'competitiveness.',
 'competitors."',
 'complex.',
 'confusing.',
 'constraints.',
 'consuming.',
 'content.',
 'controls.',
 'coordination.',
 '

In [34]:
acronyms = ["4.0", "e.g.", "etc.", "i.e."]

synonyms = {
    "I4.0": "Industry 4.0",
    "i4.0": "Industry 4.0",
    "Industry4.0": "Industry 4.0",
    "i.e": "i.e.",
}

# To check: "4.o", "E.O.L", "H&EY.", "LBBC.", "ROI.", "SAGE."

In [35]:
lemmatizer = WordNetLemmatizer()

# Found initially that some lemmatized stop words slipped through the elimination step, so including these in order to catch both the original and the lemmatized forms.
# For example, "has" and "have" are in the stop words list, but these are lemmatized to "ha" which is not.
lemmatized_stops = set([lemmatizer.lemmatize(word) for word in stopwords.words('english')] + stopwords.words('english'))

TODO: assessment of acronyms? Could search for all words (i.e., non-whitespace between whitespace) that contains full stops. Perhaps also words that end in a full stop, where the next character is lower case.

TODO: explore use of `nltk.tokenize.punkt.PunktParameters` to handle acronyms, etc that include "." characters.

In [36]:
# These acronyms and abbreviations will be expanded into their full forms.
acronyms_abbreviations = {
    'I4.0': 'industry 4.0',
    'tech': 'technology',
}

# Here we ensure that "4.0" is kept together as a word token: this should occur often together with "Industry".
# TODO confirm that this is working as expected (or is the CountVectorizer undoing the preprocessing?)
regexpr = "4\\.0|[\\w']+"
re_tokenizer = RegexpTokenizer(regexpr)

In [37]:
def preprocess_text(text):
    cleaned_text = text.replace("[REDACTED]", "")
    for short_form, full_form in acronyms_abbreviations.items():
        matches = re.findall(f'({short_form})[\\s|{re.escape(string.punctuation)}]', cleaned_text)
        for match in matches:
            cleaned_text = re.sub(re.escape(match), full_form, cleaned_text, flags=re.IGNORECASE)
    # Skipped removing punctuation as tokenization already does this.
    cleaned_text = cleaned_text.lower()
    tokenized_answer = re_tokenizer.tokenize(cleaned_text)
    lemmatized_answer = [lemmatizer.lemmatize(word) for word in tokenized_answer]
    final_answer = [lemma for lemma in lemmatized_answer if lemma not in lemmatized_stops]
    return " ".join(final_answer)

In [38]:
# responses_df['Cleaned Response'] = responses_df.apply(lambda row: preprocess_text(row['Response']), axis=1)
sentences_df['Cleaned Sentence'] = sentences_df.apply(lambda row: preprocess_text(row['Sentence']), axis=1)

In [39]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,1))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [40]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})

In [41]:
features_df.sort_values('Count', ascending=False)

,Feature,Count
930,digital,1845
467,business,1821
3221,system,1272
3256,technology,1134
1968,management,1031
...,...,...
13,150,1
15,1992,1
16,1m,1
17,1or,1


Inspection of the 30 most frequent words[1] suggests that this approach will not yield many useful concepts. Instead, we look at bigrams (word pairs) and trigrams (triples) as manual inspection of a sample of responses suggested this may be more fruitful (e.g., "digital strategy" rather than "digital" or "strategy" occuring without the other).

[1] Excluding stop words

In [42]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,3))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [43]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})
features_df['N-Gram'] = features_df['Feature'].apply(lambda feature: len(re_tokenizer.tokenize(feature)))

In [44]:
features_df.sort_values('Count', ascending=False)

,Feature,Count,N-Gram
12204,digital,1845,1
5227,business,1821,1
40245,system,1272,1
41344,technology,1134,1
25434,management,1031,1
...,...,...,...
46051,ytheir supplier,1,2
46052,ytheir supplier employee,1,3
46054,zero account,1,2
46055,zero account one,1,3


# Expert supervision
By manually inspecting the N-grams above, starting with the most frequent and working down the list, the following N-grams were identified as significant and grouped into concepts.

(So far assessed the first 6 pages - 25 results per page)

## TODO
* Visualise how these concepts map onto the different questions... would expect to see "recent innovation" to be within "Whether the business is already investing/adopting/utilising Industry 4.0 Tech..." and/or "Summary of the identified problems..."

In [45]:
concepts = {
    'Access to finance': ['investment', 'cost', 'invested'],
	  'Access to skills': ['skill', 'knowledge', 'training'],
    'Strategic alignment': ['strategy', 'productivity', 'efficiency', 'digital strategy', 'digital transformation', 'data collection', 'business growth', 'strategic'],
    'Recent innovation': [],
    'Attitudes to innovation': ['understanding', 'understand'],
    'External support/advice': ['university']
}

In [49]:
keyword = 'digital strategy'
sentences_df[f'count:{keyword}'] = sentences_df['Cleaned Sentence'].apply(lambda s: s.count(keyword))
sentences_df.groupby('Question').agg({f'count:{keyword}': 'sum'}).sort_values(f'count:{keyword}', ascending=False)

,count:digital strategy
Question,
Summary review of Edge Digital diagnostic report & current state and key improvement areas,112
"Summary of the identified problems, including Gap Analysis",104
Level of current Strategic Digital Skills/knowledge in the business,29
What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?,19
"Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples",3
Level of current Technical Digital Skills/knowledge in the business,1


In [47]:
concept = 'Access to finance'
sentences_df[f'Count:{concept}'] = sentences_df['Cleaned Sentence'].apply(lambda s: sum([s.count(keyword) for keyword in concepts[concept]]))
sentences_df.groupby('Question').agg({f'Count:{concept}': 'sum'}).sort_values(f'Count:{concept}', ascending=False)

,Count:Access to finance
Question,
What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?,572
Summary review of Edge Digital diagnostic report & current state and key improvement areas,242
"Summary of the identified problems, including Gap Analysis",227
"Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples",140
Level of current Technical Digital Skills/knowledge in the business,95
Level of current Strategic Digital Skills/knowledge in the business,65


# Information extraction

In [ ]:
model_name = "pborchert/BusinessBERT"
pipe = transformers.pipeline("fill-mask", model=model_name)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

In [ ]:
recent_innovation_questions = ['Summary of the identified problems, including Gap Analysis', 'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples']
prompt = f"This company has a {tokenizer.special_tokens_map['mask_token']} record of recent innovation and technology adoption."
# prompt = Ask the BERT to give a DRS for the firm. Need to clearly define the DRS first and ask for a numeric score.

Test case: try approach with just one firm and one concept

In [ ]:
client_1_df = responses_df[responses_df['Client ID'] == "f3fff05d-1a28-e8f3-c5f6-670d1d0797e3"]
client_1_recent_innovation = " ".join(client_1_df[client_1_df['Question'].isin(recent_innovation_questions)]['Response'].tolist())
client_1_recent_innovation

In [ ]:
mask_prompt = f"{client_1_recent_innovation}. {prompt}"

In [ ]:
mask_output = pipe(mask_prompt)
output_df = pd.DataFrame(mask_output)
output_df

Now generalise across all firms for the same concept. Note that we need to be very careful about response lengths, as the maximum number of tokens that a BERT analysis can handle in one query is 512. Some individual responses breach this limit.

In [ ]:
recent_innovation_df = responses_df[responses_df['Question'].isin(recent_innovation_questions)].groupby(['Client ID'])['Response'].apply('. '.join).to_frame().reset_index()
recent_innovation_df = recent_innovation_df.rename(columns={'Response': 'Combined Responses'})
recent_innovation_df['Combined Responses Len'] = recent_innovation_df['Combined Responses'].apply(lambda response: len(tokenizer.tokenize(response)))
recent_innovation_df

In [ ]:
n_prompt_tokens = len(tokenizer.tokenize(prompt))
n_prompt_tokens

In [ ]:
bert_max_tokens = 512
tokens_buffer = 32
max_chunk_tokens = bert_max_tokens - n_prompt_tokens - tokens_buffer
max_chunk_tokens

In [ ]:
recent_innovation_df['Split Responses'] = recent_innovation_df['Combined Responses'].apply(lambda response: chunk_text_sentencewise(response, max_chunk_tokens))
recent_innovation_df

In [ ]:
recent_innovation_df = recent_innovation_df[['Client ID', 'Split Responses']].explode('Split Responses') \
  .rename(columns={'Split Responses': 'Response Chunk'})
recent_innovation_df['Chunk Number'] = recent_innovation_df.groupby('Client ID').cumcount()
recent_innovation_df

TODO just run for a smaller subset (10-20)
* Manual verification
* Validation strategy: check how other BERT models perform - are the results consistent?

In [ ]:
responses_as_prompts = [f"{response}. {prompt}" for response in recent_innovation_df['Response Chunk'].to_list()]
all_outputs = pipe(responses_as_prompts)

In [ ]:
recent_innovation_df['Pipeline Outputs'] = all_outputs
recent_innovation_df

In [ ]:
recent_innovation_df = recent_innovation_df.explode('Pipeline Outputs')
recent_innovation_df['Token Str'] = recent_innovation_df['Pipeline Outputs'].apply(lambda output: output['token_str'])
recent_innovation_df['Token'] = recent_innovation_df['Pipeline Outputs'].apply(lambda output: output['token'])
recent_innovation_df['Score'] = recent_innovation_df['Pipeline Outputs'].apply(lambda output: output['score'])
recent_innovation_df = recent_innovation_df.drop(columns=['Pipeline Outputs'])
recent_innovation_df

In [ ]:
recent_innovation_df = pd.merge(
    recent_innovation_df,
    roadmaps_df[['Client ID', 'Current Digital Readiness Score (refer to PAS:1040)']].rename(columns={'Current Digital Readiness Score (refer to PAS:1040)': 'Current DRS'}),
    how='left',
    on='Client ID'
    )
recent_innovation_df

In [ ]:
token_strings = recent_innovation_df['Token Str'].unique().tolist()
token_strings.sort()
token_strings

In [ ]:
fig_data = [recent_innovation_df[recent_innovation_df['Token Str'] == ts]['Current DRS'].dropna() for ts in token_strings]
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.violinplot(fig_data)
ax.set_xticks(range(1, len(token_strings)+1), labels=token_strings, rotation=45)
ax.set_xlabel('Token String')
ax.set_ylabel('Current DRS')

In [ ]:
fig_df = recent_innovation_df[['Token Str', 'Current DRS', 'Score']].dropna()
fig_df['x'] = fig_df['Token Str'].apply(lambda ts: token_strings.index(ts) + 1)
fig_df['x'] = rand_jitter(fig_df['x'].to_numpy(), stdev=0.1)
fig_df

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.scatter(fig_df['x'], fig_df['Current DRS'], s=fig_df['Score'])
ax.set_xticks(range(1, len(token_strings)+1), labels=token_strings, rotation=45)
ax.set_xlabel('Token String')
ax.set_ylabel('Current DRS')

In [ ]:
token_strings.remove('track')
fig_df = recent_innovation_df[recent_innovation_df['Token Str'].isin(token_strings)][['Token Str', 'Current DRS', 'Score']].dropna()
fig_df['x'] = fig_df['Token Str'].apply(lambda ts: token_strings.index(ts) + 1)
fig_df['x'] = rand_jitter(fig_df['x'].to_numpy(), stdev=0.1)
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.scatter(fig_df['x'], fig_df['Current DRS'], s=fig_df['Score'])
ax.set_xticks(range(1, len(token_strings)+1), labels=token_strings, rotation=45)
ax.set_xlabel('Token String')
ax.set_ylabel('Current DRS')

In [ ]:
token_strings.remove('proven')
fig_df = recent_innovation_df[recent_innovation_df['Token Str'].isin(token_strings)][['Token Str', 'Current DRS', 'Score']].dropna()
fig_df['x'] = fig_df['Token Str'].apply(lambda ts: token_strings.index(ts) + 1)
fig_df['x'] = rand_jitter(fig_df['x'].to_numpy(), stdev=0.1)
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.scatter(fig_df['x'], fig_df['Current DRS'], s=fig_df['Score'])
ax.set_xticks(range(1, len(token_strings)+1), labels=token_strings, rotation=45)
ax.set_xlabel('Token String')
ax.set_ylabel('Current DRS')

In [ ]:
token_strings.remove('long')
fig_df = recent_innovation_df[recent_innovation_df['Token Str'].isin(token_strings)][['Token Str', 'Current DRS', 'Score']].dropna()
fig_df['x'] = fig_df['Token Str'].apply(lambda ts: token_strings.index(ts) + 1)
fig_df['x'] = rand_jitter(fig_df['x'].to_numpy(), stdev=0.1)
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.scatter(fig_df['x'], fig_df['Current DRS'], s=fig_df['Score'])
ax.set_xticks(range(1, len(token_strings)+1), labels=token_strings, rotation=45)
ax.set_xlabel('Token String')
ax.set_ylabel('Current DRS')

In [ ]:
token_results_df = recent_innovation_df.groupby('Token Str').agg({
    'Token Str': 'count',
    'Score': ['mean', 'min', 'max'],
    'Current DRS': ['mean', 'min', 'max']
    }).reset_index()
token_results_df